In [ ]:
"""
Task 6: Async RAG Pipeline with Cross-Encoder Reranking
Real FastAPI + FAISS + SentenceTransformers pipeline. Needs `pip install
fastapi faiss-cpu sentence-transformers uvicorn` and downloaded embedding
models to actually serve requests.
"""

import asyncio
import numpy as np
import faiss
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer, CrossEncoder

app = FastAPI()

bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")          # fast, coarse retrieval
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")  # slow, precise rerank

documents = [
    "Transformers use self-attention to model relationships between tokens.",
    "FAISS is a library for efficient similarity search of dense vectors.",
    "Retrieval-Augmented Generation grounds LLM answers in retrieved documents.",
]
doc_embeddings = bi_encoder.encode(documents, normalize_embeddings=True)

index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(np.array(doc_embeddings))

class Query(BaseModel):
    question: str
    top_k: int = 5
    final_k: int = 2

async def retrieve_candidates(question, top_k):
    q_emb = await asyncio.to_thread(bi_encoder.encode, [question], normalize_embeddings=True)
    scores, idxs = index.search(np.array(q_emb), top_k)
    return [documents[i] for i in idxs[0]]

async def rerank(question, candidates, final_k):
    pairs = [[question, c] for c in candidates]
    scores = await asyncio.to_thread(cross_encoder.predict, pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
    return ranked[:final_k]

async def synthesize_answer(question, top_chunks):
    context = "\n".join(c for c, _ in top_chunks)
    # In production this calls the LLM API asynchronously
    return f"[LLM answer using context]\nQuestion: {question}\nContext:\n{context}"

@app.post("/query")
async def query_endpoint(q: Query):
    candidates = await retrieve_candidates(q.question, q.top_k)
    reranked = await rerank(q.question, candidates, q.final_k)
    answer = await synthesize_answer(q.question, reranked)
    return {"answer": answer, "sources": [c for c, _ in reranked]}

# Run with: uvicorn task6:app --reload

---
## Task 6: Async RAG Pipeline with Cross-Encoder Reranking